# Car Model Recognition — Kaggle Training
BIL462 | Group: Junkers

**Dataset:** Add `rickyyyyyyy/torchvision-stanford-cars` as Kaggle dataset input before running.

In [ ]:
import subprocess, sys

subprocess.run(['pip', 'install', '-q', 'timm'], check=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/umutgokmen/ann-project', '/kaggle/working/ann-project'], check=True)
sys.path.insert(0, '/kaggle/working/ann-project')
print('Setup complete')

In [ ]:
import os
from pathlib import Path

DATASET_INPUT = Path('/kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars')
WORK_DIR = Path('/kaggle/working/ann-project')

print('Dataset contents:', list(DATASET_INPUT.iterdir())[:10])

In [ ]:
import yaml

config_path = WORK_DIR / 'configs' / 'config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Override paths for Kaggle
cfg['data']['dataset_dir'] = str(DATASET_INPUT)
cfg['data']['num_workers'] = 2
cfg['logging']['checkpoint_dir'] = '/kaggle/working/checkpoints'
cfg['logging']['log_dir'] = '/kaggle/working/logs'

# Kaggle T4/P100 — batch size 64 fits fine
cfg['training']['batch_size'] = 64
cfg['training']['epochs'] = 40
cfg['training']['amp'] = True

# Save modified config
kaggle_config_path = '/kaggle/working/config_kaggle.yaml'
with open(kaggle_config_path, 'w') as f:
    yaml.dump(cfg, f)

print('Config:')
print(f"  dataset_dir : {cfg['data']['dataset_dir']}")
print(f"  batch_size  : {cfg['training']['batch_size']}")
print(f"  epochs      : {cfg['training']['epochs']}")
print(f"  backbone    : {cfg['model']['backbone']}")

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Detect dataset layout and patch dataset.py if needed
import os
from pathlib import Path

dataset_dir = Path(cfg['data']['dataset_dir'])
contents = list(dataset_dir.iterdir())
print('Dataset root contents:')
for p in sorted(contents)[:20]:
    print(' ', p.name, '(dir)' if p.is_dir() else f'({p.stat().st_size // 1024}KB)')

In [ ]:
os.chdir(WORK_DIR)
from src.dataset import build_dataloaders
from src.model import build_model

loaders = build_dataloaders(cfg)
print(f"Train batches : {len(loaders['train'])}")
print(f"Val batches   : {len(loaders['val'])}")
print(f"Test batches  : {len(loaders['test'])}")
print(f"Classes       : {len(loaders['class_names'])}")

In [ ]:
# Quick sanity check — one batch forward pass
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg).to(device)

images, labels = next(iter(loaders['train']))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    out = model(images)
print(f'Batch forward OK | input: {images.shape} | output: {out.shape}')

In [ ]:
# Full training — runs ~40 epochs
# Estimated time: ~3-4h on T4, ~2h on P100
import subprocess
result = subprocess.run(
    ['python', 'src/train.py', '--config', kaggle_config_path],
    capture_output=False,
    cwd=str(WORK_DIR)
)
print('Training exit code:', result.returncode)

In [ ]:
# Evaluate best checkpoint on test set
result = subprocess.run(
    [
        'python', 'src/evaluate.py',
        '--config', kaggle_config_path,
        '--checkpoint', '/kaggle/working/checkpoints/best.pt',
        '--output_dir', '/kaggle/working/evaluation',
    ],
    capture_output=False,
    cwd=str(WORK_DIR)
)
print('Evaluation exit code:', result.returncode)

In [ ]:
# Show confusion matrix
from IPython.display import Image as IPImage
IPImage('/kaggle/working/evaluation/confusion_matrix.png')

In [ ]:
# List output files (checkpoints + evaluation)
import os
for root, dirs, files in os.walk('/kaggle/working'):
    dirs[:] = [d for d in dirs if d not in ['ann-project', '__pycache__']]
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'{path.replace("/kaggle/working/", "")}  ({size // 1024}KB)')